In [1]:
%pip install folium

In [2]:
import piplite
await piplite.install(['folium'])
await piplite.install(['pandas'])

In [3]:
import folium
import pandas as pd

In [4]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

In [5]:
# Download and read the `spacex_launch_geo.csv`
from js import fetch
import io

URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'
resp = await fetch(URL)
spacex_csv_file = io.BytesIO((await resp.arrayBuffer()).to_py())
spacex_df=pd.read_csv(spacex_csv_file)

In [6]:
# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


In [12]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

In [8]:
# Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
# Create a blue circle at NASA Johnson Space Center's coordinate with a icon showing its name
marker = folium.map.Marker(
    nasa_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

In [14]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

In [15]:
# Initial the map
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)
# For each launch site, add a Circle object based on its coordinate (Lat, Long) values. In addition, add Launch site name as a popup label


In [16]:
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


In [18]:
site_map= folium.Map(location=[29.559684888503615, -95.0830971930759],zoom_start=5)

folium.Marker([28.562302,-80.577356]).add_to(site_map)
folium.Circle([28.562302,-80.577356],radius=500,popup="CCAFS LC-40").add_to(site_map)
folium.Marker([28.563197,-80.576820]).add_to(site_map)
folium.Circle([28.563197,-80.576820],radius=500,popup="CCAFS SLC-40").add_to(site_map)
folium.Marker([28.573255,-80.646895]).add_to(site_map)
folium.Circle([28.573255,-80.646895],radius=500,popup="KSC LC-39A").add_to(site_map)
folium.Marker([34.632834,-120.610745]).add_to(site_map)
folium.Circle([34.632834,-120.610745],radius=500,popup="VAFB SLC-4E").add_to(site_map)


In [20]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class
46,KSC LC-39A,28.573255,-80.646895,1
47,KSC LC-39A,28.573255,-80.646895,1
48,KSC LC-39A,28.573255,-80.646895,1
49,CCAFS SLC-40,28.563197,-80.576820,1
50,CCAFS SLC-40,28.563197,-80.576820,1
51,CCAFS SLC-40,28.563197,-80.576820,0
52,CCAFS SLC-40,28.563197,-80.576820,0
53,CCAFS SLC-40,28.563197,-80.576820,0
54,CCAFS SLC-40,28.563197,-80.576820,1
55,CCAFS SLC-40,28.563197,-80.576820,0


In [19]:
site_map

In [22]:
from folium.plugins import MarkerCluster

marker_cluster = MarkerCluster()

for index, record in spacex_df.iterrows():
    if record['class'] == 1:
        marker = folium.Marker(
            [record['Lat'], record['Long']],
            icon=folium.Icon(color='green')
        )
    else:
        marker = folium.Marker(
            [record['Lat'], record['Long']],
            icon=folium.Icon(color='red')
        )

    marker.add_to(marker_cluster)

marker_cluster.add_to(site_map)

site_map

In [23]:
# Add Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

In [36]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

In [37]:
launch_site_lat = 28.562302
launch_site_lon = -80.577356

In [38]:
coastline_lat = 28.56367
coastline_lon = -80.56825

In [39]:
distance_coastline = calculate_distance(
    launch_site_lat,
    launch_site_lon,
    coastline_lat,
    coastline_lon
)

distance_coastline

0.9025056343150148

In [40]:
from folium.features import DivIcon

# Launch site coordinates
launch_site_lat = 28.562302
launch_site_lon = -80.577356

# Coastline coordinates
coastline_lat = 28.56367
coastline_lon = -80.56825

# Calculate distance
distance_coastline = calculate_distance(
    launch_site_lat,
    launch_site_lon,
    coastline_lat,
    coastline_lon
)

# Put the distance on the map
distance_marker = folium.Marker(
    [coastline_lat, coastline_lon],
    icon=DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12px; color:#d35400;"><b>%s</b></div>' %
        "{:10.2f} KM".format(distance_coastline)
    )
)

site_map.add_child(distance_marker)

# Draw line from launch site to coastline
coordinates = [
    [launch_site_lat, launch_site_lon],
    [coastline_lat, coastline_lon]
]

lines = folium.PolyLine(
    locations=coordinates,
    weight=1
)

site_map.add_child(lines)

# Display the map
site_map

In [41]:
closest_city_lat = 28.6150
closest_city_lon = -80.8080

In [42]:
# Calculate distance to the closest city
distance_city = calculate_distance(
    launch_site_lat,
    launch_site_lon,
    closest_city_lat,
    closest_city_lon
)

# Create marker showing the distance
distance_marker = folium.Marker(
    [closest_city_lat, closest_city_lon],
    icon=DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12px; color:#d35400;"><b>%s</b></div>' %
        "{:10.2f} KM".format(distance_city)
    )
)

# Add marker to map
site_map.add_child(distance_marker)

# Draw line between launch site and closest city
coordinates = [
    [launch_site_lat, launch_site_lon],
    [closest_city_lat, closest_city_lon]
]

lines = folium.PolyLine(
    locations=coordinates,
    weight=1
)

site_map.add_child(lines)

# Display map
site_map